In [1]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments
from datasets import Dataset, DatasetDict
import os

In [2]:
dataset_path = './dataset/bookcorpus/books_large_p1.txt' 

In [6]:
with open(dataset_path, 'r', encoding='utf-8') as f:
    content = f.readlines()

content = content[:15000]
train_content = content[:10000]  
eval_content = content[10000:12000] 

In [7]:
train_data = {"text": train_content}
eval_data = {"text": eval_content}


In [8]:
train_dataset = Dataset.from_dict(train_data)
eval_dataset = Dataset.from_dict(eval_data)

In [9]:
datasets = DatasetDict({
    'train': train_dataset,
    'validation': eval_dataset
})

In [10]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Add pad token to tokenizer (GPT-2 doesn't have a pad token by default)
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
tokenizer.pad_token = tokenizer.eos_token

In [11]:
def tokenize_function(examples):
    # Tokenize the input text and set labels as input_ids (for language modeling)
    tokenized = tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Apply tokenization to the dataset
tokenized_train_dataset = datasets['train'].map(tokenize_function, batched=True)
tokenized_eval_dataset = datasets['validation'].map(tokenize_function, batched=True)


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [12]:
training_args = TrainingArguments(
    output_dir="./gpt2-model",  # Output directory for checkpoints
    per_device_train_batch_size=1,  # Batch size per device
    num_train_epochs=3,  # Number of training epochs
    save_steps=500,  # Save model checkpoint every 500 steps
    save_total_limit=1,  # Limit to 1 checkpoint to save space
    logging_dir='./logs',  # Directory for logging
    learning_rate=5e-5,  # Learning rate
    weight_decay=0.01,  # Weight decay rate
    warmup_steps=500,  # Warmup steps for scheduler
    logging_steps=100,  # Log every 100 steps
    load_best_model_at_end=True,  # Load best model at the end of training
    eval_steps=500,  # Evaluate every 500 steps
    eval_strategy="steps",  # Evaluate based on steps
    gradient_accumulation_steps=4,  # Accumulate gradients over 4 steps
    fp16=True,  # Use mixed precision for lower memory usage
    remove_unused_columns=False  # Keep unused columns like 'text'
)

In [13]:
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Add special tokens to the model
model.resize_token_embeddings(len(tokenizer))

Embedding(50258, 768)

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
)

# Start training
trainer.train()

  0%|          | 0/7500 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
output_dir = './gpt2-model'
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model and tokenizer saved to {output_dir}")

In [5]:
# Step 1: Install required packages

from huggingface_hub import login
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

# Step 2: Login to Hugging Face (Run this only once)
# You will be prompted to enter your token
login()


In [8]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Step 1: Define Model Name and Storage Path
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Model ID
MODEL_DIR = "models/TinyLlama-1.1B-Chat-v1.0"      # Local storage path

# Step 2: Create Directory if it Doesn't Exist
os.makedirs(MODEL_DIR, exist_ok=True)

# Step 3: Configure `bitsandbytes` for 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,     # Enable 4-bit quantization
    bnb_4bit_use_double_quant=True,  # Use double quantization
    bnb_4bit_quant_type="nf4",  # NF4 quantization for better performance
    bnb_4bit_compute_dtype=torch.float16  # Use float16 for computation
)

# Step 4: Download and Save the Model with Configuration
print("Downloading model with 4-bit quantization...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,  # Apply BitsAndBytes 4-bit configuration
    device_map="auto"  # Automatically map layers to GPU/CPU
)

# Save the tokenizer and model locally
tokenizer.save_pretrained(MODEL_DIR)
model.save_pretrained(MODEL_DIR)

print(f"✅ Model downloaded and saved in '{MODEL_DIR}'")


✅ Model downloaded and saved in 'models/TinyLlama-1.1B-Chat-v1.0'


In [12]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load the model and tokenizer
local_model_path = "models/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(local_model_path)
model = AutoModelForCausalLM.from_pretrained(local_model_path).to("cuda")

# Prompt and timing
prompt = "The quick brown fox"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")




Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
`low_cpu_mem_usage` was None, now set to True since model is quantized.
You shouldn't move a model that is dispatched using accelerate hooks.


In [13]:
start_time = time.time()
output = model.generate(inputs['input_ids'], max_new_tokens=10)
end_time = time.time()

predicted_text = tokenizer.decode(output[0], skip_special_tokens=True)
print("Predicted text:", predicted_text)
print("Computation Time:", end_time - start_time, "seconds")

Predicted text: The quick brown fox jumps over the lazy dog.

2
Computation Time: 0.9468433856964111 seconds
